In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, StringType
from itertools import chain
from pyspark.sql import Window

# Reading From Bronze Table

In [0]:
df = spark.table("workspace.bronze.daily_foot_trafic")

In [0]:
print("rows:", df.count())
print("columns:", len(df.columns))

# Data Transform


##Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
      df = df.withColumn(field.name,trim(col(field.name)))

In [0]:
print("rows:", df.count())
print("columns:", len(df.columns))

##Standardized the date

In [0]:
(df.select(F.regexp_replace(F.regexp_replace(F.trim("date"), "[0-9]", "d"), "[A-Za-z]", "a").alias("pattern"))
   .groupBy("pattern").count().orderBy(F.desc("count")).show(truncate=False))

In [0]:
df = df.withColumn(
    "date",
    F.coalesce(
        F.try_to_date("date", "yyyy-MM-dd"),
        F.try_to_date("date", "yyyy-dd-MM")
    )
)

In [0]:
print("rows:", df.count())
print("columns:", len(df.columns))

## De-duplicates

In [0]:
dups = (df.groupBy(df.columns)
          .count()
          .filter("count > 1")
          .orderBy(F.desc("count")))

dups.show(1000, truncate=False)
print("duplicated groups:", dups.count())

In [0]:
w = Window.partitionBy(*df.columns).orderBy(F.lit(1))
df = df.withColumn("row_count", F.row_number().over(w))
df = df.filter("row_count = 1").drop("row_count") 

In [0]:
print("rows:", df.count())
print("columns:", len(df.columns))

#Write Into Silver Table

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.daily_foot_trafic")
)

In [0]:
%sql 
SELECT * FROM silver.daily_foot_trafic